
#  Semana 02 — Ejercicios de Optimización con PuLP

## Cuaderno de trabajo para estudiantes

Este notebook contiene **únicamente los planteamientos de los ejercicios**.  
El objetivo es que cada estudiante formule y programe su propia solución utilizando **PuLP en Python**.

---

##  Indicaciones generales

Para cada ejercicio se recomienda seguir esta secuencia:

1. Identificar las **variables de decisión**.
2. Determinar si son **continuas, enteras o binarias**.
3. Formular la **función objetivo**.
4. Escribir matemáticamente las **restricciones**.
5. Implementar el modelo en **PuLP**.
6. Resolver el modelo.
7. Revisar el estado de la solución.
8. Validar manualmente las restricciones.
9. Interpretar el resultado en el contexto del problema.

> 💡 No basta con obtener números. Debe justificarse por qué la solución encontrada es factible y qué significa en el contexto del problema.



##  Preparación del entorno

Utilice la siguiente celda únicamente para importar PuLP.

Si la librería no está instalada en su entorno, instálela antes de continuar.


In [ ]:

# Importar PuLP
import pulp



#  Ejercicio 1 — Dimensionamiento de infraestructura Cloud

##  Planteamiento

Una empresa debe contratar instancias de tres tipos para soportar una nueva plataforma.  
Se desea cubrir una capacidad mínima de **CPU** y **memoria RAM** al menor costo mensual posible.

### 📊 Datos

| Tipo | Costo mensual | vCPU | RAM |
|---|---:|---:|---:|
| A — Standard | $120 | 8 | 32 GB |
| B — Compute | $180 | 16 | 64 GB |
| C — High Capacity | $260 | 32 | 96 GB |

###  Condiciones

- Se requieren al menos **160 vCPU**.
- Se requieren al menos **520 GB de RAM**.
- Por resiliencia, deben contratarse al menos **3 instancias tipo C**.
- No pueden administrarse más de **15 instancias en total**.

---

##  Trabajo del estudiante

Formule y resuelva el modelo en PuLP.

Debe identificar:

- variables de decisión;
- tipo de variables;
- función objetivo;
- restricciones;
- solución óptima;
- costo mínimo;
- validación de CPU, RAM y número total de instancias;
- interpretación de la solución.


In [3]:

# EJERCICIO 1
# Escriba aquí su modelo en PuLP.
# Importamos la librería PuLP
import pulp

# ---------- 1. DATOS DEL PROBLEMA ----------

# Costos mensuales (en $) de cada tipo de instancia
costos = {"A": 120, "B": 180, "C": 260}

# Capacidad de vCPU de cada tipo de instancia
vcpu = {"A": 8, "B": 16, "C": 32}

# Capacidad de RAM (en GB) de cada tipo de instancia
ram = {"A": 32, "B": 64, "C": 96}

# Requisitos mínimos de capacidad
min_vcpu = 160
min_ram = 520

# Restricciones adicionales
min_instancias_C = 3
max_instancias_total = 15

# ---------- 2. CREAR EL MODELO ----------

# Creamos un problema de optimización llamado "Dimensionamiento_Cloud"
# pulp.LpMinimize indica que queremos MINIMIZAR el resultado (el costo)
modelo = pulp.LpProblem("Dimensionamiento_Cloud", pulp.LpMinimize)

# ---------- 3. VARIABLES DE DECISIÓN ----------

# Creamos una variable entera no negativa por cada tipo de instancia:
# Representa el número de instancias a contratar de cada tipo
x = {t: pulp.LpVariable(f"Instancias_{t}", lowBound=0, cat="Integer") 
     for t in ["A", "B", "C"]}

# ---------- 4. FUNCIÓN OBJETIVO ----------

# Le decimos al modelo que la meta es minimizar el costo mensual total
modelo += pulp.lpSum(costos[t] * x[t] for t in ["A", "B", "C"]), "Costo_mensual_total"

# ---------- 5. RESTRICCIONES ----------

# La suma de vCPU de todas las instancias debe ser al menos 160
modelo += pulp.lpSum(vcpu[t] * x[t] for t in ["A", "B", "C"]) >= min_vcpu, "Minimo_vCPU"

# La suma de RAM de todas las instancias debe ser al menos 520 GB
modelo += pulp.lpSum(ram[t] * x[t] for t in ["A", "B", "C"]) >= min_ram, "Minimo_RAM"

# Por resiliencia, se deben contratar al menos 3 instancias tipo C
modelo += x["C"] >= min_instancias_C, "Minimo_instancias_C"

# No se pueden administrar más de 15 instancias en total
modelo += pulp.lpSum(x[t] for t in ["A", "B", "C"]) <= max_instancias_total, "Maximo_instancias_total"

# ---------- 6. RESOLVER EL MODELO ----------

# Le pedimos al solver (motor matemático) que encuentre la mejor solución
modelo.solve()

# ---------- 7. MOSTRAR RESULTADOS ----------

# Estado de la solución: "Optimal" significa que sí se encontró la mejor respuesta
print("Estado:", pulp.LpStatus[modelo.status])

# Mostramos el número de instancias a contratar de cada tipo
print("\nNúmero de instancias a contratar:")
for t in ["A", "B", "C"]:
    print(f"Tipo {t}: {int(x[t].varValue)} instancias")

# Calculamos y mostramos la capacidad total obtenida
total_vcpu = sum(vcpu[t] * x[t].varValue for t in ["A", "B", "C"])
total_ram = sum(ram[t] * x[t].varValue for t in ["A", "B", "C"])
total_instancias = sum(x[t].varValue for t in ["A", "B", "C"])

print(f"\nCapacidad total obtenida:")
print(f"  vCPU: {int(total_vcpu)} (mínimo requerido: {min_vcpu})")
print(f"  RAM: {int(total_ram)} GB (mínimo requerido: {min_ram} GB)")
print(f"  Total de instancias: {int(total_instancias)} (máximo permitido: {max_instancias_total})")

# Mostramos el valor de la función objetivo (el costo mínimo logrado)
print(f"\nCosto mensual mínimo ($): {pulp.value(modelo.objective)}")

# Validación manual de restricciones
print("\nValidación de restricciones:")
print(f"  ✓ vCPU: {int(total_vcpu)} >= {min_vcpu} → {'CUMPLE' if total_vcpu >= min_vcpu else 'NO CUMPLE'}")
print(f"  ✓ RAM: {int(total_ram)} >= {min_ram} → {'CUMPLE' if total_ram >= min_ram else 'NO CUMPLE'}")
print(f"  ✓ Instancias tipo C: {int(x['C'].varValue)} >= {min_instancias_C} → {'CUMPLE' if x['C'].varValue >= min_instancias_C else 'NO CUMPLE'}")
print(f"  ✓ Total instancias: {int(total_instancias)} <= {max_instancias_total} → {'CUMPLE' if total_instancias <= max_instancias_total else 'NO CUMPLE'}")

Estado: Optimal

Número de instancias a contratar:
Tipo A: 0 instancias
Tipo B: 1 instancias
Tipo C: 5 instancias

Capacidad total obtenida:
  vCPU: 176 (mínimo requerido: 160)
  RAM: 544 GB (mínimo requerido: 520 GB)
  Total de instancias: 6 (máximo permitido: 15)

Costo mensual mínimo ($): 1480.0

Validación de restricciones:
  ✓ vCPU: 176 >= 160 → CUMPLE
  ✓ RAM: 544 >= 520 → CUMPLE
  ✓ Instancias tipo C: 5 >= 3 → CUMPLE
  ✓ Total instancias: 6 <= 15 → CUMPLE



##  Reto de ampliación

Modifique el modelo anterior considerando ahora:

- demanda mínima de **200 vCPU**;
- demanda mínima de **640 GB de RAM**;
- obligación de contratar al menos **2 instancias tipo A** por compatibilidad con software legado.

Compare el nuevo costo con el modelo original.


In [4]:
# RETO EJERCICIO 1
# Importar la librería PuLP
# Importamos la librería PuLP
import pulp

# ---------- 1. DATOS DEL PROBLEMA ----------

# Costos mensuales (en $) de cada tipo de instancia
costos = {"A": 120, "B": 180, "C": 260}

# Capacidad de vCPU de cada tipo de instancia
vcpu = {"A": 8, "B": 16, "C": 32}

# Capacidad de RAM (en GB) de cada tipo de instancia
ram = {"A": 32, "B": 64, "C": 96}

# Requisitos mínimos de capacidad (MODIFICADOS)
min_vcpu = 200
min_ram = 640

# Restricciones adicionales
min_instancias_C = 3
min_instancias_A = 2  # NUEVA: por compatibilidad con software legado
max_instancias_total = 15

# ---------- 2. CREAR EL MODELO ----------

# Creamos un problema de optimización llamado "Dimensionamiento_Cloud_Ampliado"
# pulp.LpMinimize indica que queremos MINIMIZAR el resultado (el costo)
modelo = pulp.LpProblem("Dimensionamiento_Cloud_Ampliado", pulp.LpMinimize)

# ---------- 3. VARIABLES DE DECISIÓN ----------

# Creamos una variable entera no negativa por cada tipo de instancia:
# Representa el número de instancias a contratar de cada tipo
x = {t: pulp.LpVariable(f"Instancias_{t}", lowBound=0, cat="Integer") 
     for t in ["A", "B", "C"]}

# ---------- 4. FUNCIÓN OBJETIVO ----------

# Le decimos al modelo que la meta es minimizar el costo mensual total
modelo += pulp.lpSum(costos[t] * x[t] for t in ["A", "B", "C"]), "Costo_mensual_total"

# ---------- 5. RESTRICCIONES ----------

# La suma de vCPU de todas las instancias debe ser al menos 200 (MODIFICADO)
modelo += pulp.lpSum(vcpu[t] * x[t] for t in ["A", "B", "C"]) >= min_vcpu, "Minimo_vCPU"

# La suma de RAM de todas las instancias debe ser al menos 640 GB (MODIFICADO)
modelo += pulp.lpSum(ram[t] * x[t] for t in ["A", "B", "C"]) >= min_ram, "Minimo_RAM"

# Por resiliencia, se deben contratar al menos 3 instancias tipo C
modelo += x["C"] >= min_instancias_C, "Minimo_instancias_C"

# Por compatibilidad con software legado, se deben contratar al menos 2 instancias tipo A (NUEVA)
modelo += x["A"] >= min_instancias_A, "Minimo_instancias_A"

# No se pueden administrar más de 15 instancias en total
modelo += pulp.lpSum(x[t] for t in ["A", "B", "C"]) <= max_instancias_total, "Maximo_instancias_total"

# ---------- 6. RESOLVER EL MODELO ----------

# Le pedimos al solver (motor matemático) que encuentre la mejor solución
modelo.solve()

# ---------- 7. MOSTRAR RESULTADOS ----------

# Estado de la solución: "Optimal" significa que sí se encontró la mejor respuesta
print("Estado:", pulp.LpStatus[modelo.status])

# Mostramos el número de instancias a contratar de cada tipo
print("\nNúmero de instancias a contratar:")
for t in ["A", "B", "C"]:
    print(f"Tipo {t}: {int(x[t].varValue)} instancias")

# Calculamos y mostramos la capacidad total obtenida
total_vcpu = sum(vcpu[t] * x[t].varValue for t in ["A", "B", "C"])
total_ram = sum(ram[t] * x[t].varValue for t in ["A", "B", "C"])
total_instancias = sum(x[t].varValue for t in ["A", "B", "C"])

print(f"\nCapacidad total obtenida:")
print(f"  vCPU: {int(total_vcpu)} (mínimo requerido: {min_vcpu})")
print(f"  RAM: {int(total_ram)} GB (mínimo requerido: {min_ram} GB)")
print(f"  Total de instancias: {int(total_instancias)} (máximo permitido: {max_instancias_total})")

# Mostramos el valor de la función objetivo (el costo mínimo logrado)
costo_nuevo = pulp.value(modelo.objective)
print(f"\nCosto mensual mínimo ($): {costo_nuevo}")

# Validación manual de restricciones
print("\nValidación de restricciones:")
print(f"  ✓ vCPU: {int(total_vcpu)} >= {min_vcpu} → {'CUMPLE' if total_vcpu >= min_vcpu else 'NO CUMPLE'}")
print(f"  ✓ RAM: {int(total_ram)} >= {min_ram} → {'CUMPLE' if total_ram >= min_ram else 'NO CUMPLE'}")
print(f"  ✓ Instancias tipo C: {int(x['C'].varValue)} >= {min_instancias_C} → {'CUMPLE' if x['C'].varValue >= min_instancias_C else 'NO CUMPLE'}")
print(f"  ✓ Instancias tipo A: {int(x['A'].varValue)} >= {min_instancias_A} → {'CUMPLE' if x['A'].varValue >= min_instancias_A else 'NO CUMPLE'}")
print(f"  ✓ Total instancias: {int(total_instancias)} <= {max_instancias_total} → {'CUMPLE' if total_instancias <= max_instancias_total else 'NO CUMPLE'}")

# Comparación con el modelo original
costo_original = 1480
diferencia = costo_nuevo - costo_original
porcentaje = (diferencia / costo_original) * 100

print(f"\n{'='*50}")
print(f"COMPARACIÓN CON EL MODELO ORIGINAL")
print(f"{'='*50}")
print(f"Costo modelo original:  ${costo_original}")
print(f"Costo modelo ampliado:  ${costo_nuevo}")
print(f"Diferencia:             ${diferencia} ({porcentaje:.1f}% más)")
print(f"{'='*50}")

Estado: Optimal

Número de instancias a contratar:
Tipo A: 2 instancias
Tipo B: 0 instancias
Tipo C: 6 instancias

Capacidad total obtenida:
  vCPU: 208 (mínimo requerido: 200)
  RAM: 640 GB (mínimo requerido: 640 GB)
  Total de instancias: 8 (máximo permitido: 15)

Costo mensual mínimo ($): 1800.0

Validación de restricciones:
  ✓ vCPU: 208 >= 200 → CUMPLE
  ✓ RAM: 640 >= 640 → CUMPLE
  ✓ Instancias tipo C: 6 >= 3 → CUMPLE
  ✓ Instancias tipo A: 2 >= 2 → CUMPLE
  ✓ Total instancias: 8 <= 15 → CUMPLE

COMPARACIÓN CON EL MODELO ORIGINAL
Costo modelo original:  $1480
Costo modelo ampliado:  $1800.0
Diferencia:             $320.0 (21.6% más)



#  Ejercicio 2 — Enrutamiento de tráfico entre enlaces WAN

##  Planteamiento

Un centro de datos debe distribuir **1,000 Mbps** entre tres enlaces WAN.  
Los enlaces tienen distintos costos, capacidades y latencias.

Se desea **minimizar el costo del tráfico**, pero la latencia promedio ponderada no debe superar **40 ms**.

###  Datos

| Enlace | Costo por Mbps | Capacidad máxima | Latencia |
|---|---:|---:|---:|
| L1 | $0.08 | 400 Mbps | 20 ms |
| L2 | $0.05 | 500 Mbps | 35 ms |
| L3 | $0.03 | 600 Mbps | 60 ms |

###  Condiciones

- Todo el tráfico debe ser enviado.
- El tráfico puede fraccionarse entre los enlaces.
- No debe superarse la capacidad máxima de cada enlace.
- La latencia promedio ponderada debe ser como máximo **40 ms**.

---

##  Trabajo del estudiante

Formule y resuelva el modelo en PuLP.

Debe determinar:

- variables de decisión;
- tipo de variables;
- función objetivo;
- restricción de balance;
- restricciones de capacidad;
- restricción de latencia promedio;
- costo mínimo;
- distribución óptima de tráfico;
- latencia promedio resultante.


In [5]:
# EJERCICIO 2
# Escriba aquí su modelo en PuLP.
# Importamos la librería PuLP
import pulp

# ---------- 1. DATOS DEL PROBLEMA ----------

# Tráfico total a distribuir (en Mbps)
trafico_total = 1000

# Costo por Mbps de cada enlace (en $)
costo = {"L1": 0.08, "L2": 0.05, "L3": 0.03}

# Capacidad máxima de cada enlace (en Mbps)
capacidad = {"L1": 400, "L2": 500, "L3": 600}

# Latencia de cada enlace (en ms)
latencia = {"L1": 20, "L2": 35, "L3": 60}

# Latencia promedio máxima permitida (en ms)
latencia_maxima = 40

# ---------- 2. CREAR EL MODELO ----------

# Creamos un problema de optimización llamado "Enrutamiento_WAN"
# pulp.LpMinimize indica que queremos MINIMIZAR el resultado (el costo)
modelo = pulp.LpProblem("Enrutamiento_WAN", pulp.LpMinimize)

# ---------- 3. VARIABLES DE DECISIÓN ----------

# Creamos una variable continua no negativa por cada enlace:
# Representa los Mbps a asignar a cada enlace
x = {enlace: pulp.LpVariable(f"Trafico_{enlace}", lowBound=0, cat="Continuous") 
     for enlace in ["L1", "L2", "L3"]}

# ---------- 4. FUNCIÓN OBJETIVO ----------

# Le decimos al modelo que la meta es minimizar el costo total del tráfico
modelo += pulp.lpSum(costo[e] * x[e] for e in ["L1", "L2", "L3"]), "Costo_total_trafico"

# ---------- 5. RESTRICCIONES ----------

# Restricción de balance: Todo el tráfico debe ser enviado (1000 Mbps)
modelo += pulp.lpSum(x[e] for e in ["L1", "L2", "L3"]) == trafico_total, "Balance_trafico"

# Restricciones de capacidad: No superar la capacidad máxima de cada enlace
for e in ["L1", "L2", "L3"]:
    modelo += x[e] <= capacidad[e], f"Capacidad_maxima_{e}"

# Restricción de latencia promedio ponderada:
# (x1*20 + x2*35 + x3*60) / 1000 <= 40
# Multiplicando ambos lados por 1000:
# 20*x1 + 35*x2 + 60*x3 <= 40000
modelo += pulp.lpSum(latencia[e] * x[e] for e in ["L1", "L2", "L3"]) <= latencia_maxima * trafico_total, "Latencia_promedio_maxima"

# ---------- 6. RESOLVER EL MODELO ----------

# Le pedimos al solver (motor matemático) que encuentre la mejor solución
modelo.solve()

# ---------- 7. MOSTRAR RESULTADOS ----------

# Estado de la solución: "Optimal" significa que sí se encontró la mejor respuesta
print("Estado:", pulp.LpStatus[modelo.status])

# Mostramos la distribución óptima de tráfico por enlace
print("\nDistribución óptima de tráfico:")
for e in ["L1", "L2", "L3"]:
    print(f"  {e}: {x[e].varValue:.2f} Mbps")

# Calculamos el costo total
costo_total = pulp.value(modelo.objective)
print(f"\nCosto mínimo total ($): {costo_total:.2f}")

# Calculamos la latencia promedio ponderada resultante
latencia_ponderada = sum(latencia[e] * x[e].varValue for e in ["L1", "L2", "L3"]) / trafico_total
print(f"Latencia promedio ponderada (ms): {latencia_ponderada:.2f}")

# Calculamos el tráfico total asignado
trafico_asignado = sum(x[e].varValue for e in ["L1", "L2", "L3"])
print(f"Tráfico total asignado (Mbps): {trafico_asignado:.2f}")

# Validación manual de restricciones
print("\nValidación de restricciones:")
print(f"  ✓ Balance: {trafico_asignado:.2f} == {trafico_total} → {'CUMPLE' if abs(trafico_asignado - trafico_total) < 0.01 else 'NO CUMPLE'}")

for e in ["L1", "L2", "L3"]:
    cumple = x[e].varValue <= capacidad[e]
    print(f"  ✓ Capacidad {e}: {x[e].varValue:.2f} <= {capacidad[e]} → {'CUMPLE' if cumple else 'NO CUMPLE'}")

print(f"  ✓ Latencia promedio: {latencia_ponderada:.2f} <= {latencia_maxima} → {'CUMPLE' if latencia_ponderada <= latencia_maxima else 'NO CUMPLE'}")

# Información adicional: costo por enlace
print("\nDetalle de costos por enlace:")
for e in ["L1", "L2", "L3"]:
    costo_enlace = costo[e] * x[e].varValue
    print(f"  {e}: {x[e].varValue:.2f} Mbps × ${costo[e]:.2f}/Mbps = ${costo_enlace:.2f}")

Estado: Optimal

Distribución óptima de tráfico:
  L1: 187.50 Mbps
  L2: 500.00 Mbps
  L3: 312.50 Mbps

Costo mínimo total ($): 49.38
Latencia promedio ponderada (ms): 40.00
Tráfico total asignado (Mbps): 1000.00

Validación de restricciones:
  ✓ Balance: 1000.00 == 1000 → CUMPLE
  ✓ Capacidad L1: 187.50 <= 400 → CUMPLE
  ✓ Capacidad L2: 500.00 <= 500 → CUMPLE
  ✓ Capacidad L3: 312.50 <= 600 → CUMPLE
  ✓ Latencia promedio: 40.00 <= 40 → CUMPLE

Detalle de costos por enlace:
  L1: 187.50 Mbps × $0.08/Mbps = $15.00
  L2: 500.00 Mbps × $0.05/Mbps = $25.00
  L3: 312.50 Mbps × $0.03/Mbps = $9.38



##  Reto de ampliación

1. Reduzca la latencia máxima permitida a **35 ms**.
2. Compare el nuevo costo con el problema original.
3. Luego simule una caída parcial de L2 reduciendo su capacidad a **200 Mbps**.
4. Determine si el problema sigue siendo factible.


In [6]:
# RETO EJERCICIO 2
# Importamos la librería PuLP
import pulp

# =====================================================
# ESCENARIO 1: Latencia máxima = 35 ms (original capacities)
# =====================================================

print("="*70)
print("ESCENARIO 1: Latencia máxima = 35 ms")
print("="*70)

# ---------- 1. DATOS DEL PROBLEMA ----------

trafico_total = 1000
costo = {"L1": 0.08, "L2": 0.05, "L3": 0.03}
capacidad = {"L1": 400, "L2": 500, "L3": 600}
latencia = {"L1": 20, "L2": 35, "L3": 60}
latencia_maxima = 35

# ---------- 2. CREAR EL MODELO ----------

modelo = pulp.LpProblem("Enrutamiento_WAN_35ms", pulp.LpMinimize)

# ---------- 3. VARIABLES DE DECISIÓN ----------

x = {enlace: pulp.LpVariable(f"Trafico_{enlace}", lowBound=0, cat="Continuous") 
     for enlace in ["L1", "L2", "L3"]}

# ---------- 4. FUNCIÓN OBJETIVO ----------

modelo += pulp.lpSum(costo[e] * x[e] for e in ["L1", "L2", "L3"]), "Costo_total_trafico"

# ---------- 5. RESTRICCIONES ----------

modelo += pulp.lpSum(x[e] for e in ["L1", "L2", "L3"]) == trafico_total, "Balance_trafico"

for e in ["L1", "L2", "L3"]:
    modelo += x[e] <= capacidad[e], f"Capacidad_maxima_{e}"

modelo += pulp.lpSum(latencia[e] * x[e] for e in ["L1", "L2", "L3"]) <= latencia_maxima * trafico_total, "Latencia_promedio_maxima"

# ---------- 6. RESOLVER EL MODELO ----------

modelo.solve()

# ---------- 7. MOSTRAR RESULTADOS ----------

print("\nEstado:", pulp.LpStatus[modelo.status])

if pulp.LpStatus[modelo.status] == 'Optimal':
    print("\nDistribución óptima de tráfico:")
    for e in ["L1", "L2", "L3"]:
        print(f"  {e}: {x[e].varValue:.2f} Mbps")
    
    costo_escenario1 = pulp.value(modelo.objective)
    print(f"\nCosto mínimo total ($): {costo_escenario1:.2f}")
    
    latencia_ponderada = sum(latencia[e] * x[e].varValue for e in ["L1", "L2", "L3"]) / trafico_total
    print(f"Latencia promedio ponderada (ms): {latencia_ponderada:.2f}")


# =====================================================
# ESCENARIO 2: Latencia = 35 ms + L2 con capacidad reducida a 200 Mbps
# =====================================================

print("\n" + "="*70)
print("ESCENARIO 2: Latencia = 35 ms + L2 con falla (200 Mbps)")
print("="*70)

# ---------- 1. DATOS DEL PROBLEMA (MODIFICADOS) ----------

capacidad_falla = {"L1": 400, "L2": 200, "L3": 600}

# ---------- 2. CREAR EL MODELO ----------

modelo_falla = pulp.LpProblem("Enrutamiento_WAN_Falla_L2", pulp.LpMinimize)

# ---------- 3. VARIABLES DE DECISIÓN ----------

y = {enlace: pulp.LpVariable(f"Trafico_{enlace}", lowBound=0, cat="Continuous") 
     for enlace in ["L1", "L2", "L3"]}

# ---------- 4. FUNCIÓN OBJETIVO ----------

modelo_falla += pulp.lpSum(costo[e] * y[e] for e in ["L1", "L2", "L3"]), "Costo_total_trafico"

# ---------- 5. RESTRICCIONES ----------

modelo_falla += pulp.lpSum(y[e] for e in ["L1", "L2", "L3"]) == trafico_total, "Balance_trafico"

for e in ["L1", "L2", "L3"]:
    modelo_falla += y[e] <= capacidad_falla[e], f"Capacidad_maxima_{e}"

modelo_falla += pulp.lpSum(latencia[e] * y[e] for e in ["L1", "L2", "L3"]) <= latencia_maxima * trafico_total, "Latencia_promedio_maxima"

# ---------- 6. RESOLVER EL MODELO ----------

modelo_falla.solve()

# ---------- 7. MOSTRAR RESULTADOS ----------

print("\nEstado:", pulp.LpStatus[modelo_falla.status])

if pulp.LpStatus[modelo_falla.status] == 'Optimal':
    print("\nDistribución óptima de tráfico:")
    for e in ["L1", "L2", "L3"]:
        print(f"  {e}: {y[e].varValue:.2f} Mbps")
    
    costo_escenario2 = pulp.value(modelo_falla.objective)
    print(f"\nCosto mínimo total ($): {costo_escenario2:.2f}")
    
    latencia_ponderada_falla = sum(latencia[e] * y[e].varValue for e in ["L1", "L2", "L3"]) / trafico_total
    print(f"Latencia promedio ponderada (ms): {latencia_ponderada_falla:.2f}")
    
    print("\n✓ El problema SIGUE SIENDO FACTIBLE")
else:
    print("\n✗ El problema NO ES FACTIBLE con estas restricciones")
    print("  No se puede cumplir con la latencia de 35 ms y la capacidad reducida de L2")


# =====================================================
# COMPARACIÓN FINAL DE ESCENARIOS
# =====================================================

print("\n" + "="*70)
print("COMPARACIÓN DE ESCENARIOS")
print("="*70)

costo_original = 49.38
latencia_original = 40

print(f"\n{'Parámetro':<25} {'Original':<20} {'Escenario 1':<20} {'Escenario 2':<20}")
print("-"*85)
print(f"{'Latencia máxima (ms)':<25} {latencia_original:<20} {35:<20} {35:<20}")
print(f"{'Capacidad L2 (Mbps)':<25} {500:<20} {500:<20} {200:<20}")
print(f"{'Costo total ($)':<25} {costo_original:<20.2f} {costo_escenario1:<20.2f} {'N/A':<20}")
print(f"{'Estado':<25} {'Óptimo':<20} {'Óptimo':<20} {pulp.LpStatus[modelo_falla.status]:<20}")

print("\n" + "-"*70)
print("ANÁLISIS DE CAMBIOS:")
print("-"*70)

incremento = costo_escenario1 - costo_original
porcentaje = (incremento / costo_original) * 100

print(f"\n1. Reducción de latencia (40→35 ms):")
print(f"   • Incremento de costo: ${incremento:.2f} ({porcentaje:.2f}%)")
print(f"   • Razón: Se debe usar más L1 (caro pero rápido) y menos L3 (barato pero lento)")

print(f"\n2. Falla parcial de L2 (500→200 Mbps):")
print(f"   • Estado: {'FACTIBLE - Se puede redistribuir' if pulp.LpStatus[modelo_falla.status] == 'Optimal' else 'NO FACTIBLE - Imposible cumplir restricciones'}")
print(f"   • Razón: Con menos capacidad en L2 y latencia máxima de 35 ms,")
print(f"     no existe combinación que cumpla todas las restricciones")

print("\n" + "="*70)

ESCENARIO 1: Latencia máxima = 35 ms

Estado: Optimal

Distribución óptima de tráfico:
  L1: 312.50 Mbps
  L2: 500.00 Mbps
  L3: 187.50 Mbps

Costo mínimo total ($): 55.62
Latencia promedio ponderada (ms): 35.00

ESCENARIO 2: Latencia = 35 ms + L2 con falla (200 Mbps)

Estado: Infeasible

✗ El problema NO ES FACTIBLE con estas restricciones
  No se puede cumplir con la latencia de 35 ms y la capacidad reducida de L2

COMPARACIÓN DE ESCENARIOS

Parámetro                 Original             Escenario 1          Escenario 2         
-------------------------------------------------------------------------------------
Latencia máxima (ms)      40                   35                   35                  
Capacidad L2 (Mbps)       500                  500                  200                 
Costo total ($)           49.38                55.62                N/A                 
Estado                    Óptimo               Óptimo               Infeasible          

--------------------


#  Ejercicio 3 — Portafolio de controles de ciberseguridad

##  Planteamiento

El CISO dispone de un presupuesto limitado y debe seleccionar controles de seguridad.  
Cada control tiene un costo y una puntuación estimada de reducción de riesgo.

### 📊 Datos

| Control | Costo | Reducción de riesgo |
|---|---:|---:|
| MFA | 12 | 25 |
| EDR | 20 | 30 |
| SIEM | 25 | 28 |
| PAM | 18 | 24 |
| Backup inmutable | 15 | 22 |
| Capacitación | 8 | 12 |

###  Condiciones

- El presupuesto máximo es **70**.
- SIEM solo puede implementarse si también se selecciona EDR.
- PAM requiere que MFA esté seleccionado.
- Debe elegirse al menos una medida entre **Backup inmutable** y **Capacitación**.
- Deben seleccionarse al menos **4 controles**.

---

## Trabajo del estudiante

Construya un modelo de programación binaria que **maximice la reducción total de riesgo**.

Debe incluir:

- una variable binaria por control;
- función objetivo;
- restricción presupuestaria;
- restricciones de dependencia;
- restricción de continuidad;
- número mínimo de controles;
- interpretación de los controles seleccionados.


In [7]:
# EJERCICIO 3
# Escriba aquí su modelo en PuLP.
# Importamos la librería PuLP
import pulp

# ---------- 1. DATOS DEL PROBLEMA ----------

# Controles de ciberseguridad disponibles
controles = ["MFA", "EDR", "SIEM", "PAM", "Backup", "Capacitacion"]

# Costo de cada control
costo = {
    "MFA": 12,
    "EDR": 20,
    "SIEM": 25,
    "PAM": 18,
    "Backup": 15,
    "Capacitacion": 8
}

# Reducción de riesgo de cada control
reduccion_riesgo = {
    "MFA": 25,
    "EDR": 30,
    "SIEM": 28,
    "PAM": 24,
    "Backup": 22,
    "Capacitacion": 12
}

# Presupuesto máximo disponible
presupuesto_maximo = 70

# ---------- 2. CREAR EL MODELO ----------

# Creamos un problema de optimización llamado "Portafolio_Ciberseguridad"
# pulp.LpMaximize indica que queremos MAXIMIZAR la reducción de riesgo
modelo = pulp.LpProblem("Portafolio_Ciberseguridad", pulp.LpMaximize)

# ---------- 3. VARIABLES DE DECISIÓN ----------

# Creamos una variable binaria (0 o 1) por cada control:
# 1 = se selecciona el control, 0 = no se selecciona
x = {control: pulp.LpVariable(f"Seleccionar_{control}", cat="Binary") 
     for control in controles}

# ---------- 4. FUNCIÓN OBJETIVO ----------

# Le decimos al modelo que la meta es maximizar la reducción total de riesgo
modelo += pulp.lpSum(reduccion_riesgo[c] * x[c] for c in controles), "Reduccion_total_riesgo"

# ---------- 5. RESTRICCIONES ----------

# Restricción presupuestaria: El costo total no debe superar el presupuesto
modelo += pulp.lpSum(costo[c] * x[c] for c in controles) <= presupuesto_maximo, "Presupuesto_maximo"

# Restricción de dependencia: SIEM solo puede implementarse si se selecciona EDR
# Si x_SIEM = 1, entonces x_EDR debe ser 1
# Si x_EDR = 0, entonces x_SIEM debe ser 0
modelo += x["SIEM"] <= x["EDR"], "SIEM_requiere_EDR"

# Restricción de dependencia: PAM requiere que MFA esté seleccionado
# Si x_PAM = 1, entonces x_MFA debe ser 1
# Si x_MFA = 0, entonces x_PAM debe ser 0
modelo += x["PAM"] <= x["MFA"], "PAM_requiere_MFA"

# Restricción de continuidad: Al menos una medida entre Backup inmutable y Capacitación
modelo += x["Backup"] + x["Capacitacion"] >= 1, "Al_menos_uno_Backup_Capacitacion"

# Restricción de número mínimo de controles: Se deben seleccionar al menos 4 controles
modelo += pulp.lpSum(x[c] for c in controles) >= 4, "Minimo_4_controles"

# ---------- 6. RESOLVER EL MODELO ----------

# Le pedimos al solver (motor matemático) que encuentre la mejor solución
modelo.solve()

# ---------- 7. MOSTRAR RESULTADOS ----------

# Estado de la solución: "Optimal" significa que sí se encontró la mejor respuesta
print("Estado:", pulp.LpStatus[modelo.status])

# Mostramos los controles seleccionados
print("\nControles seleccionados:")
controles_seleccionados = []
for c in controles:
    if x[c].varValue == 1:
        print(f"  ✓ {c:15} - Costo: ${costo[c]:2} - Reducción: {reduccion_riesgo[c]} puntos")
        controles_seleccionados.append(c)
    else:
        print(f"  ✗ {c:15} - Costo: ${costo[c]:2} - Reducción: {reduccion_riesgo[c]} puntos")

# Calculamos el costo total
costo_total = sum(costo[c] * x[c].varValue for c in controles)
print(f"\nCosto total: ${costo_total:.0f} (presupuesto máximo: ${presupuesto_maximo})")

# Mostramos el valor de la función objetivo (reducción total de riesgo)
reduccion_total = pulp.value(modelo.objective)
print(f"Reducción total de riesgo: {reduccion_total:.0f} puntos")

# Contamos el número de controles seleccionados
num_controles = sum(x[c].varValue for c in controles)
print(f"Número de controles seleccionados: {int(num_controles)} (mínimo requerido: 4)")

# Validación de restricciones
print("\nValidación de restricciones:")
print(f"  ✓ Presupuesto: ${costo_total:.0f} <= ${presupuesto_maximo} → {'CUMPLE' if costo_total <= presupuesto_maximo else 'NO CUMPLE'}")
print(f"  ✓ SIEM→EDR: SIEM={x['SIEM'].varValue}, EDR={x['EDR'].varValue} → {'CUMPLE' if x['SIEM'].varValue <= x['EDR'].varValue else 'NO CUMPLE'}")
print(f"  ✓ PAM→MFA: PAM={x['PAM'].varValue}, MFA={x['MFA'].varValue} → {'CUMPLE' if x['PAM'].varValue <= x['MFA'].varValue else 'NO CUMPLE'}")
print(f"  ✓ Backup∪Capacitacion: {x['Backup'].varValue + x['Capacitacion'].varValue} >= 1 → {'CUMPLE' if x['Backup'].varValue + x['Capacitacion'].varValue >= 1 else 'NO CUMPLE'}")
print(f"  ✓ Mínimo controles: {int(num_controles)} >= 4 → {'CUMPLE' if num_controles >= 4 else 'NO CUMPLE'}")

# Análisis de eficiencia
print("\nAnálisis de eficiencia (reducción por dólar):")
for c in controles:
    eficiencia = reduccion_riesgo[c] / costo[c]
    seleccionado = "✓" if x[c].varValue == 1 else " "
    print(f"  {seleccionado} {c:15}: {eficiencia:.2f} puntos/$")


Estado: Optimal

Controles seleccionados:
  ✓ MFA             - Costo: $12 - Reducción: 25 puntos
  ✓ EDR             - Costo: $20 - Reducción: 30 puntos
  ✗ SIEM            - Costo: $25 - Reducción: 28 puntos
  ✓ PAM             - Costo: $18 - Reducción: 24 puntos
  ✓ Backup          - Costo: $15 - Reducción: 22 puntos
  ✗ Capacitacion    - Costo: $ 8 - Reducción: 12 puntos

Costo total: $65 (presupuesto máximo: $70)
Reducción total de riesgo: 101 puntos
Número de controles seleccionados: 4 (mínimo requerido: 4)

Validación de restricciones:
  ✓ Presupuesto: $65 <= $70 → CUMPLE
  ✓ SIEM→EDR: SIEM=0.0, EDR=1.0 → CUMPLE
  ✓ PAM→MFA: PAM=1.0, MFA=1.0 → CUMPLE
  ✓ Backup∪Capacitacion: 1.0 >= 1 → CUMPLE
  ✓ Mínimo controles: 4 >= 4 → CUMPLE

Análisis de eficiencia (reducción por dólar):
  ✓ MFA            : 2.08 puntos/$
  ✓ EDR            : 1.50 puntos/$
    SIEM           : 1.12 puntos/$
  ✓ PAM            : 1.33 puntos/$
  ✓ Backup         : 1.47 puntos/$
    Capacitacion   : 1.50 punto


##  Reto de ampliación

Agregue las siguientes reglas:

1. SIEM y una herramienta *legacy* no pueden coexistir.
2. Si se elige **Backup inmutable**, también debe elegirse **MFA**.

Formule las desigualdades binarias correspondientes.

> Nota: si desea calcular un nuevo óptimo incluyendo una herramienta *legacy*, deberá definir también su costo y su contribución a la reducción de riesgo.


In [8]:
# RETO EJERCICIO 3
# Importamos la librería PuLP
import pulp

# =====================================================
# RETO DE AMPLIACIÓN: Nuevas restricciones agregadas
# =====================================================

# ---------- 1. DATOS DEL PROBLEMA ----------

# Controles de ciberseguridad disponibles (incluyendo herramienta legacy)
controles = ["MFA", "EDR", "SIEM", "PAM", "Backup", "Capacitacion", "Legacy"]

# Costo de cada control
costo = {
    "MFA": 12,
    "EDR": 20,
    "SIEM": 25,
    "PAM": 18,
    "Backup": 15,
    "Capacitacion": 8,
    "Legacy": 10  # Nueva herramienta legacy
}

# Reducción de riesgo de cada control
reduccion_riesgo = {
    "MFA": 25,
    "EDR": 30,
    "SIEM": 28,
    "PAM": 24,
    "Backup": 22,
    "Capacitacion": 12,
    "Legacy": 15  # Nueva herramienta legacy
}

# Presupuesto máximo disponible
presupuesto_maximo = 70

# ---------- 2. CREAR EL MODELO ----------

modelo = pulp.LpProblem("Portafolio_Ciberseguridad_Ampliado", pulp.LpMaximize)

# ---------- 3. VARIABLES DE DECISIÓN ----------

# Variables binarias por cada control (incluyendo legacy)
x = {control: pulp.LpVariable(f"Seleccionar_{control}", cat="Binary") 
     for control in controles}

# ---------- 4. FUNCIÓN OBJETIVO ----------

# Maximizar la reducción total de riesgo
modelo += pulp.lpSum(reduccion_riesgo[c] * x[c] for c in controles), "Reduccion_total_riesgo"

# ---------- 5. RESTRICCIONES ----------

# Restricción presupuestaria
modelo += pulp.lpSum(costo[c] * x[c] for c in controles) <= presupuesto_maximo, "Presupuesto_maximo"

# Restricción de dependencia: SIEM requiere EDR
modelo += x["SIEM"] <= x["EDR"], "SIEM_requiere_EDR"

# Restricción de dependencia: PAM requiere MFA
modelo += x["PAM"] <= x["MFA"], "PAM_requiere_MFA"

# Restricción de continuidad: Al menos uno entre Backup y Capacitación
modelo += x["Backup"] + x["Capacitacion"] >= 1, "Al_menos_uno_Backup_Capacitacion"

# Restricción de número mínimo de controles
modelo += pulp.lpSum(x[c] for c in controles) >= 4, "Minimo_4_controles"

# =====================================================
# NUEVAS RESTRICCIONES DEL RETO DE AMPLIACIÓN
# =====================================================

# Restricción 1: SIEM y Legacy no pueden coexistir
# Si seleccionamos SIEM (x_SIEM=1), entonces Legacy debe ser 0
# Si seleccionamos Legacy (x_Legacy=1), entonces SIEM debe ser 0
# Formulación: x_SIEM + x_Legacy <= 1
modelo += x["SIEM"] + x["Legacy"] <= 1, "SIEM_y_Legacy_mutuamente_excluyentes"

# Restricción 2: Si se elige Backup, también debe elegirse MFA
# Si x_Backup = 1, entonces x_MFA debe ser 1
# Si x_MFA = 0, entonces x_Backup debe ser 0
# Formulación: x_Backup <= x_MFA
modelo += x["Backup"] <= x["MFA"], "Backup_requiere_MFA"

# ---------- 6. RESOLVER EL MODELO ----------

modelo.solve()

# ---------- 7. MOSTRAR RESULTADOS ----------

print("="*80)
print("RETO DE AMPLIACIÓN - Portafolio de Ciberseguridad")
print("="*80)
print("\nEstado:", pulp.LpStatus[modelo.status])

# Mostramos los controles seleccionados
print("\nControles seleccionados:")
controles_seleccionados = []
for c in controles:
    if x[c].varValue == 1:
        print(f"  ✓ {c:15} - Costo: ${costo[c]:2} - Reducción: {reduccion_riesgo[c]} puntos")
        controles_seleccionados.append(c)
    else:
        print(f"  ✗ {c:15} - Costo: ${costo[c]:2} - Reducción: {reduccion_riesgo[c]} puntos")

# Calculamos el costo total
costo_total = sum(costo[c] * x[c].varValue for c in controles)
print(f"\nCosto total: ${costo_total:.0f} (presupuesto máximo: ${presupuesto_maximo})")

# Mostramos la reducción total de riesgo
reduccion_total = pulp.value(modelo.objective)
print(f"Reducción total de riesgo: {reduccion_total:.0f} puntos")

# Contamos el número de controles seleccionados
num_controles = sum(x[c].varValue for c in controles)
print(f"Número de controles seleccionados: {int(num_controles)} (mínimo requerido: 4)")

# Validación de restricciones
print("\n" + "="*80)
print("VALIDACIÓN DE RESTRICCIONES")
print("="*80)
print(f"  ✓ Presupuesto: ${costo_total:.0f} <= ${presupuesto_maximo} → {'CUMPLE' if costo_total <= presupuesto_maximo else 'NO CUMPLE'}")
print(f"  ✓ SIEM→EDR: SIEM={int(x['SIEM'].varValue)}, EDR={int(x['EDR'].varValue)} → {'CUMPLE' if x['SIEM'].varValue <= x['EDR'].varValue else 'NO CUMPLE'}")
print(f"  ✓ PAM→MFA: PAM={int(x['PAM'].varValue)}, MFA={int(x['MFA'].varValue)} → {'CUMPLE' if x['PAM'].varValue <= x['MFA'].varValue else 'NO CUMPLE'}")
print(f"  ✓ Backup∪Capacitacion: {int(x['Backup'].varValue + x['Capacitacion'].varValue)} >= 1 → {'CUMPLE' if x['Backup'].varValue + x['Capacitacion'].varValue >= 1 else 'NO CUMPLE'}")
print(f"  ✓ Mínimo controles: {int(num_controles)} >= 4 → {'CUMPLE' if num_controles >= 4 else 'NO CUMPLE'}")
print(f"  ✓ SIEM+Legacy≤1: {int(x['SIEM'].varValue + x['Legacy'].varValue)} <= 1 → {'CUMPLE' if x['SIEM'].varValue + x['Legacy'].varValue <= 1 else 'NO CUMPLE'}")
print(f"  ✓ Backup→MFA: Backup={int(x['Backup'].varValue)}, MFA={int(x['MFA'].varValue)} → {'CUMPLE' if x['Backup'].varValue <= x['MFA'].varValue else 'NO CUMPLE'}")

# Mostrar las nuevas restricciones formuladas
print("\n" + "="*80)
print("FORMULACIÓN DE NUEVAS RESTRICCIONES (Desigualdades Binarias)")
print("="*80)
print("\n1. SIEM y herramienta legacy no pueden coexistir:")
print("   x_SIEM + x_Legacy <= 1")
print("   Interpretación: Como máximo uno de los dos puede ser seleccionado")
print(f"   Verificación: {int(x['SIEM'].varValue)} + {int(x['Legacy'].varValue)} = {int(x['SIEM'].varValue + x['Legacy'].varValue)} <= 1 ✓")

print("\n2. Si se elige Backup inmutable, también debe elegirse MFA:")
print("   x_Backup <= x_MFA")
print("   Interpretación: Backup solo puede ser 1 si MFA también es 1")
print(f"   Verificación: {int(x['Backup'].varValue)} <= {int(x['MFA'].varValue)} ✓")

# Comparación con el ejercicio original
print("\n" + "="*80)
print("COMPARACIÓN: Ejercicio Original vs. Reto de Ampliación")
print("="*80)
print("\nEjercicio Original (sin nuevas restricciones):")
print("  Controles: MFA, EDR, PAM, Backup")
print("  Costo: $65")
print("  Reducción: 101 puntos")

print("\nReto de Ampliación (con nuevas restricciones):")
print(f"  Controles: {', '.join(controles_seleccionados)}")
print(f"  Costo: ${costo_total:.0f}")
print(f"  Reducción: {reduccion_total:.0f} puntos")

if "Legacy" in controles_seleccionados:
    print("\n✓ La herramienta Legacy fue seleccionada en la solución óptima")
else:
    print("\n La herramienta Legacy NO fue seleccionada (no es óptimo incluirla)")

print("="*80)


RETO DE AMPLIACIÓN - Portafolio de Ciberseguridad

Estado: Optimal

Controles seleccionados:
  ✓ MFA             - Costo: $12 - Reducción: 25 puntos
  ✓ EDR             - Costo: $20 - Reducción: 30 puntos
  ✗ SIEM            - Costo: $25 - Reducción: 28 puntos
  ✓ PAM             - Costo: $18 - Reducción: 24 puntos
  ✗ Backup          - Costo: $15 - Reducción: 22 puntos
  ✓ Capacitacion    - Costo: $ 8 - Reducción: 12 puntos
  ✓ Legacy          - Costo: $10 - Reducción: 15 puntos

Costo total: $68 (presupuesto máximo: $70)
Reducción total de riesgo: 106 puntos
Número de controles seleccionados: 5 (mínimo requerido: 4)

VALIDACIÓN DE RESTRICCIONES
  ✓ Presupuesto: $68 <= $70 → CUMPLE
  ✓ SIEM→EDR: SIEM=0, EDR=1 → CUMPLE
  ✓ PAM→MFA: PAM=1, MFA=1 → CUMPLE
  ✓ Backup∪Capacitacion: 1 >= 1 → CUMPLE
  ✓ Mínimo controles: 5 >= 4 → CUMPLE
  ✓ SIEM+Legacy≤1: 1 <= 1 → CUMPLE
  ✓ Backup→MFA: Backup=0, MFA=1 → CUMPLE

FORMULACIÓN DE NUEVAS RESTRICCIONES (Desigualdades Binarias)

1. SIEM y herramie


#  Ejercicio 4 — Distribución de respaldos entre niveles de almacenamiento

##  Planteamiento

Una organización debe almacenar **80 TB** de respaldos utilizando tres niveles: Hot, Warm y Cold.

Se desea **minimizar el costo mensual**, manteniendo una disponibilidad mínima y un tiempo promedio de recuperación aceptable.

###  Datos

| Nivel | Costo por TB | Tiempo de recuperación |
|---|---:|---:|
| Hot | $18 | 0.5 h |
| Warm | $10 | 4 h |
| Cold | $4 | 12 h |

###  Condiciones

- El total almacenado debe ser exactamente **80 TB**.
- Al menos **15 TB** deben permanecer en Hot.
- Al menos **20 TB** deben permanecer en Warm.
- Cold no puede superar **45 TB**.
- El tiempo promedio ponderado de recuperación debe ser como máximo **8 horas**.

---

##  Trabajo del estudiante

Formule y resuelva el modelo en PuLP.

Debe calcular:

- cantidad óptima de TB en cada nivel;
- costo mensual mínimo;
- tiempo promedio de recuperación;
- cumplimiento de todas las restricciones.


In [9]:
# EJERCICIO 4
# Escriba aquí su modelo en PuLP.
# Importamos la librería PuLP
import pulp

# ---------- 1. DATOS DEL PROBLEMA ----------

# Total de respaldos a almacenar (en TB)
total_tb = 80

# Costo por TB de cada nivel de almacenamiento (en $)
costo = {"Hot": 18, "Warm": 10, "Cold": 4}

# Tiempo de recuperación de cada nivel (en horas)
tiempo_recuperacion = {"Hot": 0.5, "Warm": 4, "Cold": 12}

# Límites de almacenamiento
min_hot = 15
min_warm = 20
max_cold = 45

# Tiempo promedio máximo de recuperación (en horas)
tiempo_promedio_maximo = 8

# ---------- 2. CREAR EL MODELO ----------

# Creamos un problema de optimización llamado "Distribucion_Respaldo"
# pulp.LpMinimize indica que queremos MINIMIZAR el costo
modelo = pulp.LpProblem("Distribucion_Respaldo", pulp.LpMinimize)

# ---------- 3. VARIABLES DE DECISIÓN ----------

# Creamos una variable continua no negativa por cada nivel:
# Representa los TB a almacenar en cada nivel
x = {nivel: pulp.LpVariable(f"TB_{nivel}", lowBound=0, cat="Continuous") 
     for nivel in ["Hot", "Warm", "Cold"]}

# ---------- 4. FUNCIÓN OBJETIVO ----------

# Le decimos al modelo que la meta es minimizar el costo mensual total
modelo += pulp.lpSum(costo[n] * x[n] for n in ["Hot", "Warm", "Cold"]), "Costo_mensual_total"

# ---------- 5. RESTRICCIONES ----------

# Restricción de balance: El total almacenado debe ser exactamente 80 TB
modelo += pulp.lpSum(x[n] for n in ["Hot", "Warm", "Cold"]) == total_tb, "Total_almacenado"

# Restricción de mínimo en Hot
modelo += x["Hot"] >= min_hot, "Minimo_Hot"

# Restricción de mínimo en Warm
modelo += x["Warm"] >= min_warm, "Minimo_Warm"

# Restricción de máximo en Cold
modelo += x["Cold"] <= max_cold, "Maximo_Cold"

# Restricción de tiempo promedio ponderado de recuperación:
# (0.5*x1 + 4*x2 + 12*x3) / 80 <= 8
# Multiplicando ambos lados por 80:
# 0.5*x1 + 4*x2 + 12*x3 <= 640
modelo += pulp.lpSum(tiempo_recuperacion[n] * x[n] for n in ["Hot", "Warm", "Cold"]) <= tiempo_promedio_maximo * total_tb, "Tiempo_promedio_maximo"

# ---------- 6. RESOLVER EL MODELO ----------

# Le pedimos al solver (motor matemático) que encuentre la mejor solución
modelo.solve()

# ---------- 7. MOSTRAR RESULTADOS ----------

# Estado de la solución: "Optimal" significa que sí se encontró la mejor respuesta
print("Estado:", pulp.LpStatus[modelo.status])

# Mostramos la distribución óptima de almacenamiento
print("\nDistribución óptima de respaldos:")
for n in ["Hot", "Warm", "Cold"]:
    print(f"  {n:5}: {x[n].varValue:.2f} TB")

# Calculamos el costo total
costo_total = pulp.value(modelo.objective)
print(f"\nCosto mensual mínimo ($): {costo_total:.2f}")

# Calculamos el tiempo promedio ponderado de recuperación
tiempo_ponderado = sum(tiempo_recuperacion[n] * x[n].varValue for n in ["Hot", "Warm", "Cold"]) / total_tb
print(f"Tiempo promedio ponderado de recuperación (h): {tiempo_ponderado:.2f}")

# Calculamos el total almacenado
total_almacenado = sum(x[n].varValue for n in ["Hot", "Warm", "Cold"])
print(f"Total almacenado (TB): {total_almacenado:.2f}")

# Validación manual de restricciones
print("\nValidación de restricciones:")
print(f"  ✓ Total almacenado: {total_almacenado:.2f} == {total_tb} → {'CUMPLE' if abs(total_almacenado - total_tb) < 0.01 else 'NO CUMPLE'}")
print(f"  ✓ Mínimo Hot: {x['Hot'].varValue:.2f} >= {min_hot} → {'CUMPLE' if x['Hot'].varValue >= min_hot else 'NO CUMPLE'}")
print(f"  ✓ Mínimo Warm: {x['Warm'].varValue:.2f} >= {min_warm} → {'CUMPLE' if x['Warm'].varValue >= min_warm else 'NO CUMPLE'}")
print(f"  ✓ Máximo Cold: {x['Cold'].varValue:.2f} <= {max_cold} → {'CUMPLE' if x['Cold'].varValue <= max_cold else 'NO CUMPLE'}")
print(f"  ✓ Tiempo promedio: {tiempo_ponderado:.2f} <= {tiempo_promedio_maximo} → {'CUMPLE' if tiempo_ponderado <= tiempo_promedio_maximo else 'NO CUMPLE'}")

# Información adicional: costo por nivel
print("\nDetalle de costos por nivel:")
for n in ["Hot", "Warm", "Cold"]:
    costo_nivel = costo[n] * x[n].varValue
    print(f"  {n:5}: {x[n].varValue:.2f} TB × ${costo[n]}/TB = ${costo_nivel:.2f}")

# Cálculo del tiempo promedio ponderado
print("\nCálculo del tiempo promedio ponderado:")
print(f"  (0.5×{x['Hot'].varValue:.2f} + 4×{x['Warm'].varValue:.2f} + 12×{x['Cold'].varValue:.2f}) / {total_tb}")
print(f"  = {0.5*x['Hot'].varValue + 4*x['Warm'].varValue + 12*x['Cold'].varValue:.2f} / {total_tb}")
print(f"  = {tiempo_ponderado:.2f} horas")

Estado: Optimal

Distribución óptima de respaldos:
  Hot  : 15.00 TB
  Warm : 20.00 TB
  Cold : 45.00 TB

Costo mensual mínimo ($): 650.00
Tiempo promedio ponderado de recuperación (h): 7.84
Total almacenado (TB): 80.00

Validación de restricciones:
  ✓ Total almacenado: 80.00 == 80 → CUMPLE
  ✓ Mínimo Hot: 15.00 >= 15 → CUMPLE
  ✓ Mínimo Warm: 20.00 >= 20 → CUMPLE
  ✓ Máximo Cold: 45.00 <= 45 → CUMPLE
  ✓ Tiempo promedio: 7.84 <= 8 → CUMPLE

Detalle de costos por nivel:
  Hot  : 15.00 TB × $18/TB = $270.00
  Warm : 20.00 TB × $10/TB = $200.00
  Cold : 45.00 TB × $4/TB = $180.00

Cálculo del tiempo promedio ponderado:
  (0.5×15.00 + 4×20.00 + 12×45.00) / 80
  = 627.50 / 80
  = 7.84 horas



##  Reto de ampliación

1. Elimine la restricción que limita Cold a **45 TB**.
2. Observe si la restricción de RTO se vuelve determinante.
3. Luego exija un RTO promedio máximo de **6 horas**.
4. Compare la nueva distribución y el costo.


In [ ]:
# RETO EJERCICIO 4
# Importamos la librería PuLP

import pulp

# =====================================================
# ESCENARIO 1: Sin límite de Cold (45 TB), RTO ≤ 8 horas
# =====================================================

print("="*80)
print("ESCENARIO 1: Sin límite de Cold - RTO máximo 8 horas")
print("="*80)

# ---------- 1. DATOS DEL PROBLEMA ----------

total_tb = 80
costo = {"Hot": 18, "Warm": 10, "Cold": 4}
tiempo_recuperacion = {"Hot": 0.5, "Warm": 4, "Cold": 12}

# Límites de almacenamiento (sin límite de Cold)
min_hot = 15
min_warm = 20
# max_cold = 45  # ELIMINADO

tiempo_promedio_maximo = 8  # RTO máximo

# ---------- 2. CREAR EL MODELO ----------

modelo1 = pulp.LpProblem("Distribucion_Respaldo_Sin_Limite_Cold", pulp.LpMinimize)

# ---------- 3. VARIABLES DE DECISIÓN ----------

x1 = {nivel: pulp.LpVariable(f"TB_{nivel}", lowBound=0, cat="Continuous") 
      for nivel in ["Hot", "Warm", "Cold"]}

# ---------- 4. FUNCIÓN OBJETIVO ----------

modelo1 += pulp.lpSum(costo[n] * x1[n] for n in ["Hot", "Warm", "Cold"]), "Costo_mensual_total"

# ---------- 5. RESTRICCIONES ----------

# Total almacenado debe ser exactamente 80 TB
modelo1 += pulp.lpSum(x1[n] for n in ["Hot", "Warm", "Cold"]) == total_tb, "Total_almacenado"

# Mínimo en Hot
modelo1 += x1["Hot"] >= min_hot, "Minimo_Hot"

# Mínimo en Warm
modelo1 += x1["Warm"] >= min_warm, "Minimo_Warm"

# NOTA: Se ELIMINÓ la restricción de máximo en Cold

# Tiempo promedio ponderado de recuperación ≤ 8 horas
modelo1 += pulp.lpSum(tiempo_recuperacion[n] * x1[n] for n in ["Hot", "Warm", "Cold"]) <= tiempo_promedio_maximo * total_tb, "Tiempo_promedio_maximo"

# ---------- 6. RESOLVER EL MODELO ----------

modelo1.solve()

# ---------- 7. MOSTRAR RESULTADOS ----------

print("\nEstado:", pulp.LpStatus[modelo1.status])

if pulp.LpStatus[modelo1.status] == 'Optimal':
    print("\nDistribución óptima de respaldos:")
    for n in ["Hot", "Warm", "Cold"]:
        print(f"  {n:5}: {x1[n].varValue:.2f} TB")
    
    costo_total1 = pulp.value(modelo1.objective)
    print(f"\nCosto mensual mínimo ($): {costo_total1:.2f}")
    
    tiempo_ponderado1 = sum(tiempo_recuperacion[n] * x1[n].varValue for n in ["Hot", "Warm", "Cold"]) / total_tb
    print(f"Tiempo promedio ponderado de recuperación (h): {tiempo_ponderado1:.2f}")
    
    # Verificar si la restricción de RTO es activa (determinante)
    if abs(tiempo_ponderado1 - tiempo_promedio_maximo) < 0.01:
        print("\n✓ La restricción de RTO es DETERMINANTE (se cumple exactamente)")
    else:
        print(f"\n  La restricción de RTO NO es determinante (margen: {tiempo_promedio_maximo - tiempo_ponderado1:.2f} h)")


# =====================================================
# ESCENARIO 2: Sin límite de Cold, RTO ≤ 6 horas
# =====================================================

print("\n" + "="*80)
print("ESCENARIO 2: Sin límite de Cold - RTO máximo 6 horas")
print("="*80)

# ---------- 1. DATOS DEL PROBLEMA (MODIFICADO) ----------

tiempo_promedio_maximo2 = 6  # RTO máximo reducido

# ---------- 2. CREAR EL MODELO ----------

modelo2 = pulp.LpProblem("Distribucion_Respaldo_RTO_6h", pulp.LpMinimize)

# ---------- 3. VARIABLES DE DECISIÓN ----------

x2 = {nivel: pulp.LpVariable(f"TB_{nivel}", lowBound=0, cat="Continuous") 
      for nivel in ["Hot", "Warm", "Cold"]}

# ---------- 4. FUNCIÓN OBJETIVO ----------

modelo2 += pulp.lpSum(costo[n] * x2[n] for n in ["Hot", "Warm", "Cold"]), "Costo_mensual_total"

# ---------- 5. RESTRICCIONES ----------

modelo2 += pulp.lpSum(x2[n] for n in ["Hot", "Warm", "Cold"]) == total_tb, "Total_almacenado"
modelo2 += x2["Hot"] >= min_hot, "Minimo_Hot"
modelo2 += x2["Warm"] >= min_warm, "Minimo_Warm"
# Sin límite de Cold

# Tiempo promedio ponderado de recuperación ≤ 6 horas
modelo2 += pulp.lpSum(tiempo_recuperacion[n] * x2[n] for n in ["Hot", "Warm", "Cold"]) <= tiempo_promedio_maximo2 * total_tb, "Tiempo_promedio_maximo"

# ---------- 6. RESOLVER EL MODELO ----------

modelo2.solve()

# ---------- 7. MOSTRAR RESULTADOS ----------

print("\nEstado:", pulp.LpStatus[modelo2.status])

if pulp.LpStatus[modelo2.status] == 'Optimal':
    print("\nDistribución óptima de respaldos:")
    for n in ["Hot", "Warm", "Cold"]:
        print(f"  {n:5}: {x2[n].varValue:.2f} TB")
    
    costo_total2 = pulp.value(modelo2.objective)
    print(f"\nCosto mensual mínimo ($): {costo_total2:.2f}")
    
    tiempo_ponderado2 = sum(tiempo_recuperacion[n] * x2[n].varValue for n in ["Hot", "Warm", "Cold"]) / total_tb
    print(f"Tiempo promedio ponderado de recuperación (h): {tiempo_ponderado2:.2f}")
    
    if abs(tiempo_ponderado2 - tiempo_promedio_maximo2) < 0.01:
        print("\n✓ La restricción de RTO es DETERMINANTE (se cumple exactamente)")
    else:
        print(f"\n  La restricción de RTO NO es determinante (margen: {tiempo_promedio_maximo2 - tiempo_ponderado2:.2f} h)")


# =====================================================
# COMPARACIÓN DE ESCENARIOS
# =====================================================

print("\n" + "="*80)
print("COMPARACIÓN DE ESCENARIOS")
print("="*80)

costo_original = 650  # Del ejercicio original con límite de 45 TB

# Usar variables intermedias para evitar f-strings anidadas
cold_original = 45.00
cold_esc1 = x1["Cold"].varValue if pulp.LpStatus[modelo1.status] == 'Optimal' else 0
cold_esc2 = x2["Cold"].varValue if pulp.LpStatus[modelo2.status] == 'Optimal' else 0

hot_original = 15.00
hot_esc1 = x1["Hot"].varValue if pulp.LpStatus[modelo1.status] == 'Optimal' else 0
hot_esc2 = x2["Hot"].varValue if pulp.LpStatus[modelo2.status] == 'Optimal' else 0

warm_original = 20.00
warm_esc1 = x1["Warm"].varValue if pulp.LpStatus[modelo1.status] == 'Optimal' else 0
warm_esc2 = x2["Warm"].varValue if pulp.LpStatus[modelo2.status] == 'Optimal' else 0

tiempo_original = 7.84
tiempo_esc1 = tiempo_ponderado1 if pulp.LpStatus[modelo1.status] == 'Optimal' else 0
tiempo_esc2 = tiempo_ponderado2 if pulp.LpStatus[modelo2.status] == 'Optimal' else 0

print(f"{'Parámetro':<25} {'Original':<20} {'Escenario 1':<20} {'Escenario 2':<20}")
print("-"*85)
print(f"{'Límite Cold (TB)':<25} {'45':<20} {'Sin límite':<20} {'Sin límite':<20}")
print(f"{'RTO máximo (h)':<25} {'8':<20} {'8':<20} {'6':<20}")
print(f"{'Cold (TB)':<25} {cold_original:<20.2f} {cold_esc1:<20.2f} {cold_esc2:<20.2f}")
print(f"{'Hot (TB)':<25} {hot_original:<20.2f} {hot_esc1:<20.2f} {hot_esc2:<20.2f}")
print(f"{'Warm (TB)':<25} {warm_original:<20.2f} {warm_esc1:<20.2f} {warm_esc2:<20.2f}")
print(f"{'Costo ($)':<25} {costo_original:<20.2f} {costo_total1:<20.2f} {costo_total2:<20.2f}")
print(f"{'Tiempo promedio (h)':<25} {tiempo_original:<20.2f} {tiempo_esc1:<20.2f} {tiempo_esc2:<20.2f}")

print("\n" + "-"*80)
print("ANÁLISIS DE CAMBIOS:")
print("-"*80)

if pulp.LpStatus[modelo1.status] == 'Optimal':
    diff1 = costo_total1 - costo_original
    print(f"\n1. Eliminar límite de Cold (RTO ≤ 8h):")
    print(f"   • Cambio de costo: ${diff1:+.2f} ({(diff1/costo_original)*100:+.2f}%)")
    print(f"   • Cold: {cold_original:.2f} TB → {cold_esc1:.2f} TB")
    
    if abs(tiempo_ponderado1 - 8) < 0.01:
        print(f"   • RTO es determinante")
    else:
        print(f"   • RTO no es determinante (margen: {8 - tiempo_ponderado1:.2f}h)")

if pulp.LpStatus[modelo2.status] == 'Optimal':
    diff2 = costo_total2 - costo_total1
    print(f"\n2. Reducir RTO a 6 horas:")
    print(f"   • Cambio de costo vs Escenario 1: ${diff2:+.2f} ({(diff2/costo_total1)*100:+.2f}%)")
    
    if hot_esc2 > hot_esc1:
        print(f"   • Se debe aumentar Hot y Warm")
    else:
        print(f"   • Se debe reducir Hot y Warm")
    
    if cold_esc2 < cold_esc1:
        print(f"   • Se debe reducir Cold")
    else:
        print(f"   • Se debe aumentar Cold")
    
    if abs(tiempo_ponderado2 - 6) < 0.01:
        print(f"   • RTO es determinante")
    else:
        print(f"   • RTO no es determinante")

print("="*80)

ESCENARIO 1: Sin límite de Cold - RTO máximo 8 horas

Estado: Optimal

Distribución óptima de respaldos:
  Hot  : 15.00 TB
  Warm : 20.00 TB
  Cold : 45.00 TB

Costo mensual mínimo ($): 650.00
Tiempo promedio ponderado de recuperación (h): 7.84

  La restricción de RTO NO es determinante (margen: 0.16 h)

ESCENARIO 2: Sin límite de Cold - RTO máximo 6 horas

Estado: Optimal

Distribución óptima de respaldos:
  Hot  : 15.00 TB
  Warm : 38.44 TB
  Cold : 26.56 TB

Costo mensual mínimo ($): 760.62
Tiempo promedio ponderado de recuperación (h): 6.00

✓ La restricción de RTO es DETERMINANTE (se cumple exactamente)

COMPARACIÓN DE ESCENARIOS
Parámetro                 Original             Escenario 1          Escenario 2         
-------------------------------------------------------------------------------------
Límite Cold (TB)          45                   Sin límite           Sin límite          
RTO máximo (h)            8                    8                    6                   
Col


#  Ejercicio 5 — Localización de nodos Edge y asignación de regiones

##  Planteamiento

Una compañía debe decidir qué nodos Edge abrir y a qué nodo asignar cada región de usuarios.

Abrir un nodo genera un **costo fijo**.  
Atender una región desde un nodo genera un costo asociado con distancia, latencia y tráfico.

###  Nodos disponibles

| Nodo | Capacidad | Costo fijo |
|---|---:|---:|
| N1 | 80 | 100 |
| N2 | 70 | 90 |
| N3 | 75 | 95 |

### Demandas regionales

| Región | Demanda |
|---|---:|
| R1 | 40 |
| R2 | 35 |
| R3 | 30 |
| R4 | 25 |

### Costos unitarios por región y nodo

| Región | N1 | N2 | N3 |
|---|---:|---:|---:|
| R1 | 2 | 5 | 7 |
| R2 | 4 | 2 | 6 |
| R3 | 6 | 3 | 2 |
| R4 | 7 | 5 | 2 |

###  Condiciones

- Cada región debe asignarse exactamente a **un nodo**.
- Una región solo puede asignarse a un nodo que haya sido abierto.
- La suma de las demandas asignadas a cada nodo no puede superar su capacidad.
- Las decisiones de apertura y asignación son binarias.

---

##  Trabajo del estudiante

Construya un modelo que minimice:

- costos fijos de apertura;
- más costos de servicio de las regiones.

Debe determinar:

- nodos que deben abrirse;
- asignación de cada región;
- costo total;
- utilización de capacidad por nodo.


In [13]:
# EJERCICIO 5
# Escriba aquí su modelo en PuLP.
# Importamos la librería PuLP
import pulp

# ---------- 1. DATOS DEL PROBLEMA ----------

# Nodos disponibles
nodos = ["N1", "N2", "N3"]
regiones = ["R1", "R2", "R3", "R4"]

# Capacidad y costo fijo de cada nodo
capacidad = {"N1": 80, "N2": 70, "N3": 75}
costo_fijo = {"N1": 100, "N2": 90, "N3": 95}

# Demanda de cada región
demanda = {"R1": 40, "R2": 35, "R3": 30, "R4": 25}

# Costos unitarios por región y nodo
costo_unitario = {
    ("R1", "N1"): 2, ("R1", "N2"): 5, ("R1", "N3"): 7,
    ("R2", "N1"): 4, ("R2", "N2"): 2, ("R2", "N3"): 6,
    ("R3", "N1"): 6, ("R3", "N2"): 3, ("R3", "N3"): 2,
    ("R4", "N1"): 7, ("R4", "N2"): 5, ("R4", "N3"): 2
}

# ---------- 2. CREAR EL MODELO ----------

modelo = pulp.LpProblem("Localizacion_Edge", pulp.LpMinimize)

# ---------- 3. VARIABLES DE DECISIÓN ----------

# y[j] = 1 si se abre el nodo j
y = {n: pulp.LpVariable(f"y_{n}", cat="Binary") for n in nodos}

# x[r,n] = 1 si la región r se asigna al nodo n
x = {(r, n): pulp.LpVariable(f"x_{r}_{n}", cat="Binary") for r in regiones for n in nodos}

# ---------- 4. FUNCIÓN OBJETIVO ----------

# Minimizar costo fijo + costo variable
modelo += pulp.lpSum(costo_fijo[n] * y[n] for n in nodos) + \
          pulp.lpSum(costo_unitario[(r, n)] * demanda[r] * x[(r, n)] for r in regiones for n in nodos)

# ---------- 5. RESTRICCIONES ----------

# Cada región se asigna a exactamente un nodo
for r in regiones:
    modelo += pulp.lpSum(x[(r, n)] for n in nodos) == 1

# Región solo se asigna a nodo abierto
for r in regiones:
    for n in nodos:
        modelo += x[(r, n)] <= y[n]

# Capacidad de nodos
for n in nodos:
    modelo += pulp.lpSum(demanda[r] * x[(r, n)] for r in regiones) <= capacidad[n] * y[n]

# ---------- 6. RESOLVER ----------

modelo.solve()

# ---------- 7. MOSTRAR RESULTADOS ----------

print("Estado:", pulp.LpStatus[modelo.status])

print("\nNodos abiertos:")
for n in nodos:
    if y[n].varValue == 1:
        print(f"  {n}: ABRIR")
    else:
        print(f"  {n}: NO ABRIR")

print("\nAsignaciones:")
for r in regiones:
    for n in nodos:
        if x[(r, n)].varValue == 1:
            print(f"  {r} → {n}")

print("\nCosto total:", pulp.value(modelo.objective))


Estado: Optimal

Nodos abiertos:
  N1: ABRIR
  N2: NO ABRIR
  N3: ABRIR

Asignaciones:
  R1 → N1
  R2 → N1
  R3 → N3
  R4 → N3

Costo total: 525.0



##  Reto de ampliación

Analice las siguientes modificaciones:

1. Exigir que se abran al menos **2 nodos**.
2. Exigir que se abran exactamente **2 nodos**.
3. Imponer que **R1 no pueda utilizar N3** debido a un SLA de latencia.

Compare las soluciones obtenidas.


In [14]:
# RETO EJERCICIO 5
# Importamos la librería PuLP
import pulp

# ---------- 1. DATOS DEL PROBLEMA ----------

nodos = ["N1", "N2", "N3"]
regiones = ["R1", "R2", "R3", "R4"]

capacidad = {"N1": 80, "N2": 70, "N3": 75}
costo_fijo = {"N1": 100, "N2": 90, "N3": 95}
demanda = {"R1": 40, "R2": 35, "R3": 30, "R4": 25}

costo_unitario = {
    ("R1", "N1"): 2, ("R1", "N2"): 5, ("R1", "N3"): 7,
    ("R2", "N1"): 4, ("R2", "N2"): 2, ("R2", "N3"): 6,
    ("R3", "N1"): 6, ("R3", "N2"): 3, ("R3", "N3"): 2,
    ("R4", "N1"): 7, ("R4", "N2"): 5, ("R4", "N3"): 2
}

# =====================================================
# ESCENARIO 1: Al menos 2 nodos abiertos
# =====================================================

print("="*60)
print("ESCENARIO 1: Al menos 2 nodos abiertos")
print("="*60)

modelo1 = pulp.LpProblem("Escenario1", pulp.LpMinimize)

y1 = {n: pulp.LpVariable(f"y_{n}", cat="Binary") for n in nodos}
x1 = {(r, n): pulp.LpVariable(f"x_{r}_{n}", cat="Binary") for r in regiones for n in nodos}

modelo1 += pulp.lpSum(costo_fijo[n] * y1[n] for n in nodos) + \
           pulp.lpSum(costo_unitario[(r, n)] * demanda[r] * x1[(r, n)] for r in regiones for n in nodos)

for r in regiones:
    modelo1 += pulp.lpSum(x1[(r, n)] for n in nodos) == 1

for r in regiones:
    for n in nodos:
        modelo1 += x1[(r, n)] <= y1[n]

for n in nodos:
    modelo1 += pulp.lpSum(demanda[r] * x1[(r, n)] for r in regiones) <= capacidad[n] * y1[n]

# NUEVA: Al menos 2 nodos
modelo1 += pulp.lpSum(y1[n] for n in nodos) >= 2

modelo1.solve()

print("Estado:", pulp.LpStatus[modelo1.status])
print("Nodos abiertos:", [n for n in nodos if y1[n].varValue == 1])
print("Asignaciones:")
for r in regiones:
    for n in nodos:
        if x1[(r, n)].varValue == 1:
            print(f"  {r} → {n}")
print("Costo total:", pulp.value(modelo1.objective))

# =====================================================
# ESCENARIO 2: Exactamente 2 nodos abiertos
# =====================================================

print("\n" + "="*60)
print("ESCENARIO 2: Exactamente 2 nodos abiertos")
print("="*60)

modelo2 = pulp.LpProblem("Escenario2", pulp.LpMinimize)

y2 = {n: pulp.LpVariable(f"y_{n}", cat="Binary") for n in nodos}
x2 = {(r, n): pulp.LpVariable(f"x_{r}_{n}", cat="Binary") for r in regiones for n in nodos}

modelo2 += pulp.lpSum(costo_fijo[n] * y2[n] for n in nodos) + \
           pulp.lpSum(costo_unitario[(r, n)] * demanda[r] * x2[(r, n)] for r in regiones for n in nodos)

for r in regiones:
    modelo2 += pulp.lpSum(x2[(r, n)] for n in nodos) == 1

for r in regiones:
    for n in nodos:
        modelo2 += x2[(r, n)] <= y2[n]

for n in nodos:
    modelo2 += pulp.lpSum(demanda[r] * x2[(r, n)] for r in regiones) <= capacidad[n] * y2[n]

# NUEVA: Exactamente 2 nodos
modelo2 += pulp.lpSum(y2[n] for n in nodos) == 2

modelo2.solve()

print("Estado:", pulp.LpStatus[modelo2.status])
print("Nodos abiertos:", [n for n in nodos if y2[n].varValue == 1])
print("Asignaciones:")
for r in regiones:
    for n in nodos:
        if x2[(r, n)].varValue == 1:
            print(f"  {r} → {n}")
print("Costo total:", pulp.value(modelo2.objective))

# =====================================================
# ESCENARIO 3: R1 no puede usar N3
# =====================================================

print("\n" + "="*60)
print("ESCENARIO 3: R1 no puede usar N3")
print("="*60)

modelo3 = pulp.LpProblem("Escenario3", pulp.LpMinimize)

y3 = {n: pulp.LpVariable(f"y_{n}", cat="Binary") for n in nodos}
x3 = {(r, n): pulp.LpVariable(f"x_{r}_{n}", cat="Binary") for r in regiones for n in nodos}

modelo3 += pulp.lpSum(costo_fijo[n] * y3[n] for n in nodos) + \
           pulp.lpSum(costo_unitario[(r, n)] * demanda[r] * x3[(r, n)] for r in regiones for n in nodos)

for r in regiones:
    modelo3 += pulp.lpSum(x3[(r, n)] for n in nodos) == 1

for r in regiones:
    for n in nodos:
        modelo3 += x3[(r, n)] <= y3[n]

for n in nodos:
    modelo3 += pulp.lpSum(demanda[r] * x3[(r, n)] for r in regiones) <= capacidad[n] * y3[n]

# NUEVA: R1 no puede usar N3
modelo3 += x3[("R1", "N3")] == 0

modelo3.solve()

print("Estado:", pulp.LpStatus[modelo3.status])
print("Nodos abiertos:", [n for n in nodos if y3[n].varValue == 1])
print("Asignaciones:")
for r in regiones:
    for n in nodos:
        if x3[(r, n)].varValue == 1:
            print(f"  {r} → {n}")
print("Costo total:", pulp.value(modelo3.objective))

# =====================================================
# COMPARACIÓN
# =====================================================

print("\n" + "="*60)
print("COMPARACIÓN DE ESCENARIOS")
print("="*60)
print(f"Escenario 1 (≥2 nodos): ${pulp.value(modelo1.objective):.0f}")
print(f"Escenario 2 (=2 nodos): ${pulp.value(modelo2.objective):.0f}")
print(f"Escenario 3 (R1≠N3):    ${pulp.value(modelo3.objective):.0f}")

ESCENARIO 1: Al menos 2 nodos abiertos
Estado: Optimal
Nodos abiertos: ['N1', 'N3']
Asignaciones:
  R1 → N1
  R2 → N1
  R3 → N3
  R4 → N3
Costo total: 525.0

ESCENARIO 2: Exactamente 2 nodos abiertos
Estado: Optimal
Nodos abiertos: ['N1', 'N3']
Asignaciones:
  R1 → N1
  R2 → N1
  R3 → N3
  R4 → N3
Costo total: 525.0

ESCENARIO 3: R1 no puede usar N3
Estado: Optimal
Nodos abiertos: ['N1', 'N3']
Asignaciones:
  R1 → N1
  R2 → N1
  R3 → N3
  R4 → N3
Costo total: 525.0

COMPARACIÓN DE ESCENARIOS
Escenario 1 (≥2 nodos): $525
Escenario 2 (=2 nodos): $525
Escenario 3 (R1≠N3):    $525



#  Ejercicio 6 — Dimensionamiento de agentes de CI/CD

## Planteamiento

Una plataforma DevOps necesita capacidad concurrente para pipelines Linux y Windows.

Existen tres tipos de agentes con diferentes capacidades y costos.

###  Datos

| Tipo de agente | Linux slots | Windows slots | Costo |
|---|---:|---:|---:|
| Standard | 4 | 2 | 50 |
| Linux Optimized | 8 | 0 | 70 |
| Universal | 3 | 5 | 80 |

### Condiciones

- Se requieren al menos **40 slots Linux**.
- Se requieren al menos **20 slots Windows**.
- Deben existir al menos **2 agentes Universal**.
- El equipo de operaciones puede administrar como máximo **12 agentes**.

---

##  Trabajo del estudiante

Formule y resuelva un modelo de programación entera que minimice el costo total.

Debe determinar:

- cantidad de agentes Standard;
- cantidad de agentes Linux Optimized;
- cantidad de agentes Universal;
- costo mínimo;
- slots Linux obtenidos;
- slots Windows obtenidos;
- total de agentes utilizados.


In [15]:
# EJERCICIO 6
# Escriba aquí su modelo en PuLP.
# Importamos la librería PuLP
import pulp

# ---------- 1. DATOS DEL PROBLEMA ----------

# Tipos de agentes
agentes = ["Standard", "LinuxOpt", "Universal"]

# Slots Linux por tipo de agente
linux_slots = {"Standard": 4, "LinuxOpt": 8, "Universal": 3}

# Slots Windows por tipo de agente
windows_slots = {"Standard": 2, "LinuxOpt": 0, "Universal": 5}

# Costo por tipo de agente
costo = {"Standard": 50, "LinuxOpt": 70, "Universal": 80}

# Requisitos mínimos
min_linux = 40
min_windows = 20
min_universal = 2
max_agentes = 12

# ---------- 2. CREAR EL MODELO ----------

modelo = pulp.LpProblem("Dimensionamiento_CI_CD", pulp.LpMinimize)

# ---------- 3. VARIABLES DE DECISIÓN ----------

x = {a: pulp.LpVariable(f"Agentes_{a}", lowBound=0, cat="Integer") for a in agentes}

# ---------- 4. FUNCIÓN OBJETIVO ----------

modelo += pulp.lpSum(costo[a] * x[a] for a in agentes)

# ---------- 5. RESTRICCIONES ----------

# Mínimo slots Linux
modelo += pulp.lpSum(linux_slots[a] * x[a] for a in agentes) >= min_linux

# Mínimo slots Windows
modelo += pulp.lpSum(windows_slots[a] * x[a] for a in agentes) >= min_windows

# Mínimo agentes Universal
modelo += x["Universal"] >= min_universal

# Máximo total de agentes
modelo += pulp.lpSum(x[a] for a in agentes) <= max_agentes

# ---------- 6. RESOLVER ----------

modelo.solve()

# ---------- 7. MOSTRAR RESULTADOS ----------

print("Estado:", pulp.LpStatus[modelo.status])

print("\nCantidad de agentes:")
for a in agentes:
    print(f"  {a}: {int(x[a].varValue)}")

total_linux = sum(linux_slots[a] * x[a].varValue for a in agentes)
total_windows = sum(windows_slots[a] * x[a].varValue for a in agentes)
total_agentes = sum(x[a].varValue for a in agentes)

print(f"\nSlots Linux: {int(total_linux)}")
print(f"Slots Windows: {int(total_windows)}")
print(f"Total agentes: {int(total_agentes)}")
print(f"Costo mínimo: ${pulp.value(modelo.objective):.0f}")


Estado: Optimal

Cantidad de agentes:
  Standard: 5
  LinuxOpt: 2
  Universal: 2

Slots Linux: 42
Slots Windows: 20
Total agentes: 9
Costo mínimo: $550



## Reto de ampliación

Modifique el modelo de la siguiente manera:

1. Aumente el requerimiento de Windows a **30 slots**.
2. Agregue la regla:

> Por cada 3 agentes Linux Optimized debe existir al menos 1 agente Universal.

Formule matemáticamente dicha restricción e incorpórela al modelo.

Compare el nuevo costo con el problema original.


In [ ]:
# RETO EJERCICIO 6
# Importamos la librería PuLP
import pulp

# ---------- 1. DATOS DEL PROBLEMA ----------

agentes = ["Standard", "LinuxOpt", "Universal"]
linux_slots = {"Standard": 4, "LinuxOpt": 8, "Universal": 3}
windows_slots = {"Standard": 2, "LinuxOpt": 0, "Universal": 5}
costo = {"Standard": 50, "LinuxOpt": 70, "Universal": 80}

min_linux = 40
min_windows = 30  # MODIFICADO: era 20
min_universal = 2
max_agentes = 12

# ---------- 2. CREAR EL MODELO ----------

modelo = pulp.LpProblem("CI_CD_Ampliado", pulp.LpMinimize)

# ---------- 3. VARIABLES DE DECISIÓN ----------

x = {a: pulp.LpVariable(f"Agentes_{a}", lowBound=0, cat="Integer") for a in agentes}

# ---------- 4. FUNCIÓN OBJETIVO ----------

modelo += pulp.lpSum(costo[a] * x[a] for a in agentes)

# ---------- 5. RESTRICCIONES ----------

modelo += pulp.lpSum(linux_slots[a] * x[a] for a in agentes) >= min_linux
modelo += pulp.lpSum(windows_slots[a] * x[a] for a in agentes) >= min_windows
modelo += x["Universal"] >= min_universal
modelo += pulp.lpSum(x[a] for a in agentes) <= max_agentes

# NUEVA: Por cada 3 LinuxOpt, al menos 1 Universal
# x_LinuxOpt <= 3 * x_Universal
modelo += x["LinuxOpt"] <= 3 * x["Universal"]

# ---------- 6. RESOLVER ----------

modelo.solve()

# ---------- 7. MOSTRAR RESULTADOS ----------

print("Estado:", pulp.LpStatus[modelo.status])

print("\nCantidad de agentes:")
for a in agentes:
    print(f"  {a}: {int(x[a].varValue)}")

total_linux = sum(linux_slots[a] * x[a].varValue for a in agentes)
total_windows = sum(windows_slots[a] * x[a].varValue for a in agentes)
total_agentes = sum(x[a].varValue for a in agentes)

print(f"\nSlots Linux: {int(total_linux)}")
print(f"Slots Windows: {int(total_windows)}")
print(f"Total agentes: {int(total_agentes)}")
print(f"Costo mínimo: ${pulp.value(modelo.objective):.0f}")

# Comparación
print("\n" + "="*60)
print("COMPARACIÓN")
print("="*60)
print(f"Costo original (Windows=20):  $550")
print(f"Costo nuevo (Windows=30):     ${pulp.value(modelo.objective):.0f}")
print(f"Diferencia:                   ${pulp.value(modelo.objective) - 550:+.0f}")


Estado: Optimal

Cantidad de agentes:
  Standard: 8
  LinuxOpt: 0
  Universal: 3

Slots Linux: 41
Slots Windows: 31
Total agentes: 11
Costo mínimo: $640

COMPARACIÓN
Costo original (Windows=20):  $550
Costo nuevo (Windows=30):     $640
Diferencia:                   $+90



#  Entrega sugerida

Para cada ejercicio, entregue:

- formulación matemática;
- código en PuLP;
- estado del solver;
- valores de las variables;
- valor de la función objetivo;
- comprobación de restricciones;
- interpretación breve de la solución.

> Si el estado del modelo no es `Optimal`, no interprete los valores de las variables como una solución óptima.
